In [3]:
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import association_rules
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display

In [4]:
groceries = []
with open("../Datasets/Groceries.csv") as f:groceries = f.read()
groceries = groceries.split("\n")
groceries_list = []
for i in groceries:
    groceries_list.append(i.split(','))

In [5]:
groceries_list[:10]

[['citrus fruit', 'semi-finished bread', 'margarine', 'ready soups'],
 ['tropical fruit', 'yogurt', 'coffee'],
 ['whole milk'],
 ['pip fruit', 'yogurt', 'cream cheese', 'meat spreads'],
 ['other vegetables',
  'whole milk',
  'condensed milk',
  'long life bakery product'],
 ['whole milk', 'butter', 'yogurt', 'rice', 'abrasive cleaner'],
 ['rolls/buns'],
 ['other vegetables',
  'UHT-milk',
  'rolls/buns',
  'bottled beer',
  'liquor (appetizer)'],
 ['potted plants'],
 ['whole milk', 'cereals']]

In [6]:
te = TransactionEncoder().set_output(transform="pandas")
te_df = te.fit(groceries_list).transform(groceries_list)
te_df

,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,baby food,bags,baking powder,bathroom cleaner,beef,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9830,False,False,False,False,False,False,False,False,False,True,...,False,False,False,True,False,False,False,True,False,False
9831,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
9832,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
9833,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [9]:
def gen_rules (min_support, min_confidence):
    itemsets = apriori(te_df, min_support=min_support, use_colnames=True)
    rules = association_rules(itemsets, metric='confidence', min_threshold=min_confidence)
    rules = rules[['antecedents','consequents','support','confidence','lift']]
    rel_rules = rules[rules["lift"]>1]
    return rel_rules.sort_values(by=['confidence','lift','support'],ascending=False)

In [6]:
interact(
    gen_rules, 
    min_support = widgets.FloatSlider(value=0.01, min=0, max=1, step=0.01, description='Min Support:'), 
    min_confidence = widgets.FloatSlider(value=0.01, min=0, max=1, step=0.01, description='Min Confidence:')
)

interactive(children=(FloatSlider(value=0.01, description='Min Support:', max=1.0, step=0.01), FloatSlider(val…

<function __main__.gen_rules(min_support, min_confidence)>

In [10]:
def gen_rules(min_support, min_confidence):
    # Change min_spport to min_support here:
    itemsets = apriori(te_df, min_support=min_support, use_colnames=True)
    
    rules = association_rules(itemsets, metric='confidence', min_threshold=min_confidence)
    
    if rules.empty:
        print("No rules found for these thresholds.")
        return
        
    rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    rel_rules = rules[rules["lift"] > 1]
    
    display(rel_rules.sort_values(by=['confidence', 'lift', 'support'], ascending=False))

In [11]:
interact(
    gen_rules, 
    min_support=widgets.FloatSlider(value=0.2, min=0.01, max=1.0, step=0.01, description='Min Support:'), 
    min_confidence=widgets.FloatSlider(value=0.5, min=0.01, max=1.0, step=0.01, description='Min Confidence:')
);

interactive(children=(FloatSlider(value=0.2, description='Min Support:', max=1.0, min=0.01, step=0.01), FloatS…

In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Define your widgets
slider_support = widgets.FloatSlider(value=0.2, min=0.01, max=1.0, step=0.01, description='Min Support:')
slider_confidence = widgets.FloatSlider(value=0.5, min=0.01, max=1.0, step=0.01, description='Min Confidence:')

# 2. Create an output area widget
out = widgets.Output()

def update_rules(change):
    with out:
        clear_output(wait=True)
        # Compute itemsets and rules based on current slider values
        itemsets = apriori(te_df, min_support=slider_support.value, use_colnames=True)
        rules = association_rules(itemsets, metric='confidence', min_threshold=slider_confidence.value)
        
        if rules.empty:
            print("No rules found for these thresholds.")
            return
            
        rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
        rel_rules = rules[rules["lift"] > 1]
        
        # Display the filtered and sorted DataFrame
        display(rel_rules.sort_values(by=['confidence', 'lift', 'support'], ascending=False))

# 3. Bind the sliders to the update function
slider_support.observe(update_rules, names='value')
slider_confidence.observe(update_rules, names='value')

# 4. Display widgets and output container together
display(widgets.VBox([slider_support, slider_confidence, out]))

# Trigger the initial display
update_rules(None)

In [10]:
gen_rules (0.6,0.7)

ValueError: The input DataFrame `df` containing the frequent itemsets is empty.